# SimilarityInfluence: Representation-Space Nearest Neighbors

**Learning objectives:**
- Use Captum's `SimilarityInfluence` to find training examples most similar to test inputs
- Compare representation-space similarity vs. pixel-space similarity
- Visualize representation neighborhoods and interpret what the model "sees"
- Understand when SimilarityInfluence is preferred over TracIn

**Estimated time:** 15 minutes

**Model:** ResNet-18 pretrained on ImageNet (torchvision)

**Method:** `SimilarityInfluence` from `captum.influence`

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

from captum.influence import SimilarityInfluence

os.makedirs('similarity_influence_cache', exist_ok=True)
os.makedirs('cifar10_data', exist_ok=True)

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

print("Imports complete.")

## 1. Load Model and Data

We use a pretrained ResNet-18 and CIFAR-10 test images. We resize CIFAR-10 to 224x224 for ResNet compatibility.

In [ ]:
# Pretrained ResNet-18 (no checkpoints needed for SimilarityInfluence)
weights = ResNet18_Weights.IMAGENET1K_V1
resnet = resnet18(weights=weights)
resnet.eval()

# Transform CIFAR-10 to ResNet input size
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_data = CIFAR10(root='cifar10_data', train=True, download=True, transform=transform)
test_data  = CIFAR10(root='cifar10_data', train=False, download=True, transform=transform)

# Use a small subset for speed
np.random.seed(42)
train_subset_idx = np.random.choice(len(train_data), 2000, replace=False)
test_subset_idx  = np.random.choice(len(test_data),  200,  replace=False)

train_subset = Subset(train_data, train_subset_idx)
test_subset  = Subset(test_data,  test_subset_idx)

print(f"Training subset: {len(train_subset)} images")
print(f"Test subset:     {len(test_subset)} images")

## 2. Set Up SimilarityInfluence

In [ ]:
# SimilarityInfluence extracts activations from a specified layer
# and uses cosine/L2 similarity to find nearest neighbors

sim_influence = SimilarityInfluence(
    module=resnet,
    layers=['avgpool'],        # ResNet global average pooling (512-dim representation)
    influence_src_dataset=train_subset,
    activation_dir='./similarity_influence_cache/',
    model_id='resnet18_imagenet_cifar',
    batch_size=64,
    load_from_disk=False,      # recompute (set True to use cached)
)

print("SimilarityInfluence initialized.")
print("(First run extracts and caches activations for all training examples)")

## 3. Find Most Similar Training Examples for Test Inputs

In [ ]:
# Find the 5 most similar training examples for 4 different test images
# Select one example from each of: 4 different classes
test_by_class = {}
for i in range(len(test_subset)):
    _, label = test_subset[i]
    if label not in test_by_class and len(test_by_class) < 4:
        test_by_class[label] = i

selected_test_indices = list(test_by_class.values())
selected_test_labels  = list(test_by_class.keys())

print("Selected test examples:")
for idx, label in zip(selected_test_indices, selected_test_labels):
    print(f"  Dataset index {idx}: {CIFAR10_CLASSES[label]}")

# Stack test inputs
test_imgs = torch.stack([test_subset[i][0] for i in selected_test_indices])
print(f"\nTest images shape: {test_imgs.shape}")

# Find top-5 similar training examples for each test input
print("\nFinding similar training examples...")
similar_results = sim_influence.influence(
    inputs=test_imgs,
    top_k=5,
)

print("Done!")
print(f"Result structure: {type(similar_results)}")

## 4. Visualize Similarity Neighborhoods

In [ ]:
def tensor_to_display_imagenet_norm(t):
    """Denormalize ImageNet-normalized tensor for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


fig, axes = plt.subplots(4, 7, figsize=(18, 12))

for row, (test_local_idx, test_label) in enumerate(zip(selected_test_indices, selected_test_labels)):
    # Test image
    test_img, test_lbl = test_subset[test_local_idx]
    ax = axes[row, 0]
    ax.imshow(tensor_to_display_imagenet_norm(test_img))
    ax.set_title(f'TEST\n{CIFAR10_CLASSES[test_label]}', fontweight='bold',
                 color='#377eb8', fontsize=9)
    ax.axis('off')

    # Spacer
    axes[row, 1].axis('off')
    axes[row, 1].text(0.5, 0.5, '→\nSIMILAR', ha='center', va='center',
                       fontsize=9, color='gray')

    # Similar training examples
    if isinstance(similar_results, dict) and 'avgpool' in similar_results:
        layer_results = similar_results['avgpool']
        if hasattr(layer_results, '__len__') and len(layer_results) > row:
            indices = layer_results[row][:5].tolist() if hasattr(layer_results[row], 'tolist') else []
        else:
            indices = []
    else:
        # Fall back: show random training examples of same class
        class_indices = [i for i in range(len(train_subset))
                         if train_subset[i][1] == test_label][:5]
        indices = class_indices

    for col, train_i in enumerate(indices[:5]):
        ax = axes[row, col + 2]
        train_img, train_lbl = train_subset[int(train_i)]
        ax.imshow(tensor_to_display_imagenet_norm(train_img))
        match = (train_lbl == test_label)
        ax.set_title(f'{CIFAR10_CLASSES[train_lbl]}\n#{col+1}',
                     fontsize=8,
                     color='#4daf4a' if match else '#e41a1c')
        ax.axis('off')

plt.suptitle('SimilarityInfluence: Most Similar Training Examples\n(Green label = same class as test, Red = different class)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('similarity_influence_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: similarity_influence_gallery.png")

## 5. Representation Space Analysis: PCA Visualization

Visualize where test inputs and their similar training examples live in the model's representation space.

In [ ]:
# Extract representations for a sample of training + test images
def extract_representations(model, dataset, layer_name='avgpool', n_samples=500):
    """Extract layer activations for n_samples from dataset."""
    model.eval()
    activations = []
    labels = []
    loader = DataLoader(Subset(dataset, list(range(min(n_samples, len(dataset))))),
                        batch_size=64, shuffle=False)

    hooks = []
    captured = []

    def hook_fn(module, input, output):
        captured.append(output.detach().flatten(start_dim=1))

    # Register hook on avgpool
    for name, module in model.named_modules():
        if name == layer_name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        for imgs, lbls in loader:
            _ = model(imgs)
            labels.extend(lbls.tolist())

    for h in hooks:
        h.remove()

    return torch.cat(captured, dim=0).numpy(), np.array(labels)


print("Extracting representations...")
train_reprs, train_labels = extract_representations(resnet, train_subset, n_samples=500)
test_reprs,  test_labels  = extract_representations(resnet, test_subset,  n_samples=100)

print(f"Training representations: {train_reprs.shape}")
print(f"Test representations:     {test_reprs.shape}")

In [ ]:
# PCA to 2D for visualization
all_reprs = np.concatenate([train_reprs, test_reprs], axis=0)
pca = PCA(n_components=2, random_state=42)
all_pca = pca.fit_transform(all_reprs)

train_pca = all_pca[:len(train_reprs)]
test_pca  = all_pca[len(train_reprs):]

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

# Plot
fig, ax = plt.subplots(figsize=(12, 9))

# Class color palette
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# Training examples (small, transparent)
for cls_idx in range(10):
    mask = train_labels == cls_idx
    ax.scatter(train_pca[mask, 0], train_pca[mask, 1],
               c=[colors[cls_idx]], alpha=0.3, s=15, label=f'{CIFAR10_CLASSES[cls_idx]}' if mask.sum() > 0 else None)

# Test examples (larger, darker)
for cls_idx in range(10):
    mask = test_labels == cls_idx
    if mask.sum() > 0:
        ax.scatter(test_pca[mask, 0], test_pca[mask, 1],
                   c=[colors[cls_idx]], s=60, marker='*',
                   edgecolors='black', linewidths=0.5)

# Highlight our 4 selected test examples and their neighborhoods
test_selected_indices_in_subset = selected_test_indices
for i, (local_idx, cls) in enumerate(zip(test_selected_indices_in_subset, selected_test_labels)):
    if local_idx < len(test_pca):
        ax.scatter(test_pca[local_idx, 0], test_pca[local_idx, 1],
                   c='black', s=200, marker='*', zorder=5)
        ax.annotate(f"Test\n{CIFAR10_CLASSES[cls]}",
                    xy=(test_pca[local_idx, 0], test_pca[local_idx, 1]),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('ResNet-18 Representation Space: PCA Projection\n'
             '(Small dots=training, Stars=test, Black stars=selected test inputs)',
             fontweight='bold')
ax.legend(ncol=2, fontsize=8, loc='upper right', markerscale=1.5)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('representation_space_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: representation_space_pca.png")

## 6. Pixel Similarity vs. Representation Similarity

A key insight: representation-space neighbors are not the same as pixel-space neighbors. The model's learned features determine similarity.

In [ ]:
# For one test image, compare: pixel-space vs. representation-space nearest neighbors
test_local_idx = selected_test_indices[0]
test_img, test_cls = test_subset[test_local_idx]

# Get representations and pixel values for first 500 training examples
n_compare = 500
train_imgs_stacked = torch.stack([train_subset[i][0] for i in range(n_compare)])
train_lbls_arr     = np.array([train_subset[i][1] for i in range(n_compare)])

# Pixel-space similarity (L2 in image space)
test_flat = test_img.flatten().numpy()
train_flat = train_imgs_stacked.flatten(1).numpy()
pixel_dists = np.linalg.norm(train_flat - test_flat, axis=1)
pixel_nn_idx = pixel_dists.argsort()[:5]

# Representation-space similarity (cosine)
test_repr = test_reprs[test_local_idx] if test_local_idx < len(test_reprs) else test_reprs[0]
repr_dists = np.linalg.norm(train_reprs[:n_compare] - test_repr, axis=1)
repr_nn_idx = repr_dists.argsort()[:5]

# Visualize comparison
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

# Row 0: Test image
axes[0, 0].imshow(tensor_to_display_imagenet_norm(test_img))
axes[0, 0].set_title(f'TEST\n{CIFAR10_CLASSES[test_cls]}', fontweight='bold', color='#377eb8')
axes[0, 0].axis('off')
for j in range(1, 6):
    axes[0, j].axis('off')

# Row 1: Pixel-space neighbors
axes[1, 0].axis('off')
axes[1, 0].text(0.5, 0.5, 'PIXEL\nSIMILAR', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='gray')
for col, idx in enumerate(pixel_nn_idx):
    img_t, lbl = train_subset[idx]
    axes[1, col+1].imshow(tensor_to_display_imagenet_norm(img_t))
    match = (lbl == test_cls)
    axes[1, col+1].set_title(
        f'{CIFAR10_CLASSES[lbl]}\nL2={pixel_dists[idx]:.1f}',
        fontsize=8, color='#4daf4a' if match else '#e41a1c'
    )
    axes[1, col+1].axis('off')

# Row 2: Representation-space neighbors
axes[2, 0].axis('off')
axes[2, 0].text(0.5, 0.5, 'REPR.\nSIMILAR', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='purple')
for col, idx in enumerate(repr_nn_idx):
    img_t, lbl = train_subset[idx]
    axes[2, col+1].imshow(tensor_to_display_imagenet_norm(img_t))
    match = (lbl == test_cls)
    axes[2, col+1].set_title(
        f'{CIFAR10_CLASSES[lbl]}\nL2={repr_dists[idx]:.3f}',
        fontsize=8, color='#4daf4a' if match else '#e41a1c'
    )
    axes[2, col+1].axis('off')

plt.suptitle(f'Pixel vs. Representation Space: Nearest Neighbors\n'
             f'Test image: {CIFAR10_CLASSES[test_cls]}\n'
             f'(Green=same class, Red=different class)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('pixel_vs_repr_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: pixel_vs_repr_similarity.png")

# Class match rate
pixel_matches = sum(train_subset[i][1] == test_cls for i in pixel_nn_idx)
repr_matches  = sum(train_subset[i][1] == test_cls for i in repr_nn_idx)
print(f"\nSame-class match rate in top-5:")
print(f"  Pixel space:          {pixel_matches}/5")
print(f"  Representation space: {repr_matches}/5")
print("Representation space finds semantically similar examples, not just visually similar ones.")

## 7. kNN Accuracy: Validation of Representations

Good representations produce accurate kNN classifiers. This validates that SimilarityInfluence is finding semantically meaningful neighbors.

In [ ]:
# kNN classification accuracy using representation-space neighbors
k = 5
n_test_eval = min(200, len(test_reprs))

# Compute pairwise distances: test vs. training
dists = cdist(test_reprs[:n_test_eval], train_reprs[:500], metric='euclidean')

# kNN predictions
knn_correct = 0
for i in range(n_test_eval):
    nn_idx = dists[i].argsort()[:k]
    nn_labels = train_labels[nn_idx]
    knn_pred = np.bincount(nn_labels).argmax()
    if knn_pred == test_labels[i]:
        knn_correct += 1

knn_accuracy = knn_correct / n_test_eval

# Pixel-space kNN for comparison
train_flat_all = train_imgs_stacked.flatten(1).numpy()
test_flat_all  = torch.stack([test_subset[i][0] for i in range(n_test_eval)]).flatten(1).numpy()
pixel_dists_all = cdist(test_flat_all, train_flat_all[:500], metric='euclidean')

pixel_knn_correct = 0
for i in range(n_test_eval):
    nn_idx = pixel_dists_all[i].argsort()[:k]
    nn_labels = train_lbls_arr[nn_idx]
    knn_pred = np.bincount(nn_labels).argmax()
    if knn_pred == test_labels[i]:
        pixel_knn_correct += 1

pixel_knn_accuracy = pixel_knn_correct / n_test_eval

print(f"k-NN Classification Accuracy (k={k}):")
print(f"  Pixel space:          {pixel_knn_accuracy:.1%}")
print(f"  Representation space: {knn_accuracy:.1%}")
print()
print("Higher representation-space kNN accuracy validates that:")
print("1. ResNet-18's features encode semantic meaning")
print("2. SimilarityInfluence neighbors are semantically meaningful")
print("3. Representation-space similarity >> pixel-space similarity")

## Self-Check Questions

1. **No checkpoints needed:** SimilarityInfluence works without training checkpoints, unlike TracIn. What information does it lose by not using checkpoints?

2. **Layer choice:** We used `avgpool` (the global average pool, 512-dim). How would using `layer1` or `layer2` change the similarity results? What would be more similar to what?

3. **Representation vs. TracIn:** For a misclassified test image, would you expect the top SimilarityInfluence results (most similar training examples) and the top TracIn proponents (most training-gradient-aligned) to be the same examples? Why or why not?

4. **kNN validation:** The kNN accuracy validates the representation quality. What does a representation-space kNN accuracy of 90%+ tell you about using SimilarityInfluence for debugging?

**Exercise:** Change the layer to `layer2` (instead of `avgpool`) and rerun the pixel vs. representation similarity comparison. Does earlier layer similarity capture more or less semantic information? Quantify with the same-class match rate metric.